# Chronos Embeddings + MLP Classifier

Train a small MLP head on top of frozen Chronos T5 embeddings of IMU motion sequences. The first cell defines `separar_sin_repeticiones`, which converts the per-row feature CSV into per-(label, subject) numpy arrays; the second cell loads the pretrained Chronos pipeline; the third wires both together and runs k-fold evaluation.

## Sequence builder

In [ ]:
import os
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Any

import numpy as np
import pandas as pd


def separar_sin_repeticiones(
    csv_path: str,
    feature_cols: List[str],
    label_col: str = "label",
    subject_col: str = "subject",
    min_len: int = 5,                            # descarta secuencias demasiado cortas
) -> Tuple[List[np.ndarray], List[Any], List[Any], pd.DataFrame]:
    """
    Supuestos:
      - No existe columna de tiempo; el orden de las filas representa el tiempo.
      - No hay repeticiones: cada (subject, label) aparece exactamente una vez.
      - Las secuencias pueden tener distinta longitud (n variable).
      - feature_cols contiene exactamente las 14 características de interés.

    Devuelve:
      - X_list: lista de arrays (n_i × 14)
      - y_labels: lista de labels (longitud = número de secuencias)
      - subjects: lista de subjects alineada con y_labels/X_list
      - meta: DataFrame con metadatos (ruta, n_rows, n_features, etc.)
    """
    df = pd.read_csv(csv_path)

    # Validaciones básicas
    missing = [c for c in [label_col, subject_col] if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")
    for c in feature_cols:
        if c not in df.columns:
            raise ValueError(f"Falta columna de característica en CSV: {c}")

    # Orden estable por (subject, label) y por el índice original (como proxy de tiempo)
    df = df.reset_index().rename(columns={"index": "_orig_idx"})
    df = df.sort_values([subject_col, label_col, "_orig_idx"]).reset_index(drop=True)

    # Agrupar por (subject, label)
    groups = df.groupby([subject_col, label_col], sort=True)

    X_list, y_labels, subjects = [], [], []
    records = []


    for (subject, label), g in groups:
        # Extraer matriz n × 14 (sin NaN/Inf)
        G = g[feature_cols].to_numpy(dtype=np.float32)
        # Limpiar filas no numéricas si existieran
        mask_finite_rows = np.isfinite(G).all(axis=1)
        G = G[mask_finite_rows]

        if len(G) < min_len:
            continue

        # Acumular salidas
        X_list.append(G)             # n_i × 14
        y_labels.append(label)
        subjects.append(subject)

        records.append({
            "label": label,
            "subject": subject,
            "n_rows": G.shape[0],
            "n_features": G.shape[1],
            
        })

    meta = pd.DataFrame.from_records(records).sort_values(["label", "subject"]).reset_index(drop=True)

    return X_list, y_labels, subjects, meta


# ====== EJEMPLO DE USO ======
FEATURES = [
    "arm_flex_r","arm_add_r","arm_rot_r","elbow_flex_r","pro_sup_r",
    "wrist_flex_r","wrist_dev_r","arm_flex_l","arm_add_l","arm_rot_l",
    "elbow_flex_l","pro_sup_l","wrist_flex_l","wrist_dev_l"
]

X_list, y_labels, subjects, meta = separar_sin_repeticiones(
csv_path="general_db_imu.csv",
feature_cols=FEATURES,
label_col="label",
subject_col="subject",
min_len=5
)
print(len(X_list), "secuencias extraídas")
print(meta.head())


## Load Chronos pipeline

In [ ]:
import numpy as np
import torch
from chronos import ChronosPipeline

# === Config ===
MODEL_ID = "amazon/chronos-t5-large"   # cambia a la variante que prefieras
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

# === Cargar pipeline Chronos ===
pipe = ChronosPipeline.from_pretrained(MODEL_ID, device_map=DEVICE)
pipe.model.eval()
for p in pipe.model.parameters():
    p.requires_grad = False

# d_model (dimensión del embedding)
d = getattr(pipe.model.config, "d_model", None)
print(f"d_model (embedding dim): {d}")

embeds_per_seq = []  # guardará un vector [d] por secuencia

with torch.no_grad():
    for X in X_list:           # cada X: [n_i, 14]
        # 1) Construir batch de 14 series (una por feature), limpiando NaN/Inf
        series = []
        for j in range(X.shape[1]):
            s = np.asarray(X[:, j], dtype=np.float32)
            s = np.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0)  # robustez básica
            series.append(torch.from_numpy(s))

        # 2) Embedding en batch de las 14 series → [14, T, d]
        E, _ = pipe.embed(series)          # E: [num_series=14, T=n_i, d]
        # 3) Pooling temporal (media sobre T) → [14, d]
        V = E.mean(dim=1)
        # 4) Media entre features → [d]
        z = V.mean(dim=0)

        embeds_per_seq.append(z.cpu().numpy())  # vector [d]

# 5) Apilar todas las secuencias → [N, d] y transponer → [d, N]
Z = np.stack(embeds_per_seq, axis=0)   # [N, d] (N ≈ 249)
Z = Z.T                                 # [d, N]

print("Z shape:", Z.shape)  # debería imprimir (d, 249) si N=249


## Train and evaluate the classifier

In [ ]:
import os
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Any
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from chronos import ChronosPipeline
import matplotlib.pyplot as plt


def separar_sin_repeticiones(
    csv_path: str,
    feature_cols: List[str],
    label_col: str = "label",
    subject_col: str = "subject",
    min_len: int = 5,
) -> Tuple[List[np.ndarray], List[Any], List[Any], pd.DataFrame]:
    """
    Extrae secuencias de tiempo por (subject, label) sin repeticiones.
    """
    df = pd.read_csv(csv_path)

    # Validaciones básicas
    missing = [c for c in [label_col, subject_col] if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")
    for c in feature_cols:
        if c not in df.columns:
            raise ValueError(f"Falta columna de característica en CSV: {c}")

    # Orden estable por (subject, label) y por el índice original
    df = df.reset_index().rename(columns={"index": "_orig_idx"})
    df = df.sort_values([subject_col, label_col, "_orig_idx"]).reset_index(drop=True)

    # Agrupar por (subject, label)
    groups = df.groupby([subject_col, label_col], sort=True)
    X_list, y_labels, subjects = [], [], []
    records = []

    for (subject, label), g in groups:
        # Extraer matriz n × features
        G = g[feature_cols].to_numpy(dtype=np.float32)
        # Limpiar filas no numéricas
        mask_finite_rows = np.isfinite(G).all(axis=1)
        G = G[mask_finite_rows]

        if len(G) < min_len:
            continue

        X_list.append(G)
        y_labels.append(label)
        subjects.append(subject)

        records.append({
            "label": label,
            "subject": subject,
            "n_rows": G.shape[0],
            "n_features": G.shape[1],
        })

    meta = pd.DataFrame.from_records(records).sort_values(["label", "subject"]).reset_index(drop=True)
    return X_list, y_labels, subjects, meta


def extract_chronos_embeddings(X_list: List[np.ndarray], model_id: str = "amazon/chronos-t5-mini") -> np.ndarray:
    """
    Extrae embeddings usando Chronos para cada secuencia.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Cargar pipeline Chronos
    pipe = ChronosPipeline.from_pretrained(model_id, device_map=device)
    pipe.model.eval()
    for p in pipe.model.parameters():
        p.requires_grad = False

    # Obtener dimensión del embedding
    d_model = getattr(pipe.model.config, "d_model", 512)
    print(f"Dimensión del embedding (d_model): {d_model}")

    embeds_per_seq = []
    
    with torch.no_grad():
        for i, X in enumerate(X_list):
            if i % 50 == 0:
                print(f"Procesando secuencia {i+1}/{len(X_list)}")
            
            # Construir batch de series (una por feature)
            series = []
            for j in range(X.shape[1]):  # Para cada feature
                s = np.asarray(X[:, j], dtype=np.float32)
                s = np.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0)
                series.append(torch.from_numpy(s))

            try:
                # Embedding en batch
                E, _ = pipe.embed(series)  # [num_features, T, d_model]
                
                # Pooling temporal (media sobre T)
                V = E.mean(dim=1)  # [num_features, d_model]
                
                # Media entre features
                z = V.mean(dim=0)  # [d_model]
                
                embeds_per_seq.append(z.cpu().numpy())
                
            except Exception as e:
                print(f"Error procesando secuencia {i}: {e}")
                # Usar embedding cero como fallback
                embeds_per_seq.append(np.zeros(d_model, dtype=np.float32))

    return np.stack(embeds_per_seq, axis=0)  # [N, d_model]


class SignLanguageDataset(Dataset):
    """Dataset para embeddings de lenguaje de señas"""
    def __init__(self, embeddings: np.ndarray, labels: np.ndarray):
        self.embeddings = torch.FloatTensor(embeddings)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.embeddings)
    
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


class SignLanguageClassifier(nn.Module):
    """Clasificador simple para embeddings de Chronos"""
    def __init__(self, input_dim: int, num_classes: int, hidden_dim: int = 256, dropout: float = 0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)


def train_classifier(embeddings: np.ndarray, labels: List[str], subjects: List[str], 
                    test_size: float = 0.2, random_state: int = 42, epochs: int = 100):
    """
    Entrena el clasificador con validación estratificada por subject.
    """
    # Codificar labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(labels)
    num_classes = len(label_encoder.classes_)
    
    print(f"Número de clases: {num_classes}")
    print(f"Clases: {label_encoder.classes_}")
    
    # Split estratificado - asegurándonos de que cada subject aparezca solo en train o test
    unique_subjects = list(set(subjects))
    subjects_train, subjects_test = train_test_split(
        unique_subjects, test_size=test_size, random_state=random_state
    )
    
    # Crear máscaras para train/test basadas en subjects
    train_mask = np.array([s in subjects_train for s in subjects])
    test_mask = np.array([s in subjects_test for s in subjects])
    
    X_train, X_test = embeddings[train_mask], embeddings[test_mask]
    y_train, y_test = y_encoded[train_mask], y_encoded[test_mask]
    
    print(f"Conjunto de entrenamiento: {len(X_train)} muestras")
    print(f"Conjunto de prueba: {len(X_test)} muestras")
    
    # Crear datasets y dataloaders
    train_dataset = SignLanguageDataset(X_train, y_train)
    test_dataset = SignLanguageDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    # Modelo
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SignLanguageClassifier(
        input_dim=embeddings.shape[1],
        num_classes=num_classes,
        hidden_dim=256
    ).to(device)
    
    # Optimizador y función de pérdida
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    
    # Entrenamiento
    train_losses, test_accuracies = [], []
    best_acc = 0
    
    for epoch in range(epochs):
        # Entrenamiento
        model.train()
        train_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Evaluación
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                _, predicted = torch.max(outputs.data, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        
        test_acc = 100 * correct / total
        avg_train_loss = train_loss / len(train_loader)
        
        train_losses.append(avg_train_loss)
        test_accuracies.append(test_acc)
        
        scheduler.step(avg_train_loss)
        
        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), 'best_model.pth')
        
        if epoch % 10 == 0:
            print(f'Epoch {epoch}: Train Loss: {avg_train_loss:.4f}, Test Acc: {test_acc:.2f}%')
    
    # Cargar mejor modelo y evaluación final
    model.load_state_dict(torch.load('best_model.pth'))
    model.eval()
    
    # Predicciones finales
    y_pred = []
    y_true = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs, 1)
            y_pred.extend(predicted.cpu().numpy())
            y_true.extend(batch_y.cpu().numpy())
    
    # Reporte de clasificación
    y_true_labels = label_encoder.inverse_transform(y_true)
    y_pred_labels = label_encoder.inverse_transform(y_pred)
    
    print("\n=== RESULTADOS FINALES ===")
    print(f"Mejor accuracy: {best_acc:.2f}%")
    print("\nReporte de clasificación:")
    print(classification_report(y_true_labels, y_pred_labels))
    
    return model, label_encoder, train_losses, test_accuracies, y_true_labels, y_pred_labels





train_classifier(z, y, subjects: List[str], 
                    test_size: float = 0.2, random_state: int = 42, epochs: int = 100)